# Prepare ADU Viewer Data

Converts your cleaned parcel data + classified raster into the two formats
`index.html` reads directly:

- `data/parcels.geojson` — parcels reprojected to EPSG:4326 (what web maps expect)
- `data/raster_overlay.png` + `data/raster_bounds.json` — your classified raster,
  downsampled, reprojected, and colorized, with the corner coordinates MapLibre
  needs to place it

**Memory note:** this version reads the raster with a *decimated* read
(`out_shape` + `Resampling.mode`) instead of loading the full-resolution array.
For a citywide raster at fine NAIP/LiDAR resolution, a full-res read can be
several GB and will crash a Jupyter kernel outright (no traceback — the OS
just kills the process). Decimating first also happens to produce the right
output size for a browser `image` overlay, which shouldn't be citywide at
full resolution anyway.

**Requirements:**
```
pip install geopandas rasterio pillow numpy matplotlib --break-system-packages
```


In [1]:
import json
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from PIL import Image
import matplotlib.pyplot as plt

## Config

In [8]:
# the GeoPackage from the clean_parcels notebook — using the .gpkg (not .shp) here
# because several of the renamed fields (BuildingSqft, ADU_backParking, MinADUSpace)
# get truncated to 10 characters in a shapefile's .dbf, and we want the real names
PARCEL_SOURCE = "/Users/gusstephens/Desktop/ADU Project/SLC_Parcels_ADU_Potential_Landcover.gpkg"

RASTER_TIF = "/Users/gusstephens/Desktop/ADU Project/fiveband_raster_noncornerbackyards_RFclass_masked_V8_postSampleEdits.tif"

OUTPUT_DIR = "data"   # relative to wherever this notebook lives — put it inside adu-viewer/ or adjust this

# longest side of the exported overlay image, in pixels. Most GPUs/browsers
# cap a single texture at 8192px — going much past that risks the image
# silently failing to render on some devices (older phones especially).
# 8000 is a good ceiling: noticeably sharper than 5000 while staying under
# that limit. MapLibre's `image` source still isn't meant for citywide
# full-native-resolution rasters, but this gets you close to the practical max.
MAX_OUTPUT_DIM = 8000

## Step 1 — Parcels: load, reproject, export

Loads the cleaned parcel layer and writes it straight to GeoJSON in WGS84.
No field renaming needed here since that already happened in `clean_parcels`.

In [9]:
gdf = gpd.read_file(PARCEL_SOURCE)
print(f"Loaded {len(gdf)} parcels")
print(f"CRS: {gdf.crs}")
print(f"Columns: {list(gdf.columns)}")

for col in ["ADU_yn", "InTransit", "District"]:
    if col in gdf.columns:
        print(f"\n{col} unique values: {sorted(gdf[col].dropna().unique().tolist())}")

Loaded 34535 parcels
CRS: EPSG:3566
Columns: ['ZONING', 'ZONING_NAM', 'PARCEL_ID', 'PARCEL_ADD', 'BuildingSqft', 'GrassSqft', 'PavedSqft', 'SoilSqft', 'TreeSqft', 'WaterSqft', 'MinADUSpace', 'ADU_yn', 'ADU_backParking', 'InTransit', 'District', 'geometry']

ADU_yn unique values: [0.0, 1.0]

InTransit unique values: [0, 1]

District unique values: ['Council District 1', 'Council District 2', 'Council District 3', 'Council District 4', 'Council District 5', 'Council District 6', 'Council District 7']


In [11]:
gdf_wgs84 = gdf.to_crs(epsg=4326)

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = f"{OUTPUT_DIR}/parcels.geojson"
gdf_wgs84["geometry"] = gdf_wgs84["geometry"].simplify(0.00001, preserve_topology=True)
gdf_wgs84.to_file(out_path, driver="GeoJSON", COORDINATE_PRECISION=6)
print(f"Wrote {out_path} ({len(gdf_wgs84)} parcels)")

minx, miny, maxx, maxy = gdf_wgs84.total_bounds
print(f"Map center for index.html: [{(minx+maxx)/2:.5f}, {(miny+maxy)/2:.5f}]")

Wrote data/parcels.geojson (34535 parcels)
Map center for index.html: [-111.87337, 40.75659]


## Step 2 — Raster: inspect metadata only (no pixel data read yet)

Just opens the file header — this is instant and safe regardless of raster
size. Confirms band count and native resolution before we touch any pixels.

In [ ]:
with rasterio.open(RASTER_TIF) as src:
    print(f"Bands: {src.count}")
    print(f"Size: {src.width} x {src.height} px  ({src.width * src.height / 1e6:.1f} million pixels/band)")
    print(f"CRS: {src.crs}")
    print(f"Dtype: {src.dtypes}")
    print(f"Pixel size: {src.res}")
    est_gb = (src.width * src.height * src.count) / 1e9  # rough, assumes 1 byte/px
    print(f"\nRough full-resolution read size: ~{est_gb:.2f} GB per byte-of-dtype "
          f"(multiply by dtype size, e.g. x4 for float32) — this is why we don't do a full read.")

## Step 3 — Decimated read for inspection

Reads every band at a small, fixed preview size (not full resolution) using
`Resampling.mode`, which picks the most common value in each downsampled
block — the right choice for categorical/class data, since it won't invent
blended values the way average/bilinear would. This is enough to tell which
band holds discrete class codes vs. continuous predictor values.

In [ ]:
PREVIEW_DIM = 800  # small and fast, just for band identification

with rasterio.open(RASTER_TIF) as src:
    scale = min(1.0, PREVIEW_DIM / max(src.width, src.height))
    prev_w, prev_h = max(1, int(src.width * scale)), max(1, int(src.height * scale))

    for b in range(1, src.count + 1):
        preview = src.read(b, out_shape=(prev_h, prev_w), resampling=Resampling.mode)
        n_unique = len(np.unique(preview))
        print(f"Band {b}: min={preview.min()}, max={preview.max()}, unique values in preview={n_unique}"
              + ("  <- looks like discrete classes" if n_unique <= 20 else ""))

## Step 4 — Confirm the classification band + class colors + preview

Set `CLASS_BAND` to whichever band Step 3 flagged as discrete (likely 1, but
confirm). Then check the printed unique values below against `CLASS_COLORS`
— adjust the keys if your classifier used different integer codes than
guessed here. The six classes assumed below match the order of your
`VALUE_1`–`VALUE_6` sqft fields (Building, Grass, Paved, Soil, Tree, Water).

In [ ]:
CLASS_BAND = 1

with rasterio.open(RASTER_TIF) as src:
    scale = min(1.0, MAX_OUTPUT_DIM / max(src.width, src.height))
    out_w, out_h = max(1, int(src.width * scale)), max(1, int(src.height * scale))
    print(f"Native size: {src.width}x{src.height} -> decimated output size: {out_w}x{out_h}")

    classified_src_crs = src.read(CLASS_BAND, out_shape=(out_h, out_w), resampling=Resampling.mode)
    print(f"Unique values at output resolution: {np.unique(classified_src_crs)}")

# map each class integer -> RGBA color. Adjust keys to match the unique values printed above.
CLASS_COLORS = {
    1: (255, 0, 0, 255),      # Building/Structure - red
    2: (0, 255, 0, 255),      # ILV (grass/lawn) - bright green
    3: (0, 0, 0, 255),        # Impervious (asphalt/concrete) - black
    4: (153, 140, 86, 255),   # NLV/Soil - tan/olive
    5: (34, 102, 34, 255),    # Trees - dark green
    6: (0, 0, 255, 255),      # Water - blue
}
CLASS_LABELS = {1: "Building/Structure", 2: "ILV (grass/lawn)", 3: "Impervious (asphalt/concrete)", 4: "NLV/Soil", 5: "Trees", 6: "Water"}

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(classified_src_crs, cmap="tab10")
ax.set_title(f"Band {CLASS_BAND} — decimated to {out_w}x{out_h}, raw class values")
ax.set_axis_off()
plt.show()

## Step 5 — Reproject the decimated array, colorize, export

Now working with a small array (thousands of pixels per side, not tens of
thousands), so reprojection is fast and safe. `Resampling.nearest` here
because we're reprojecting an already-categorical array and don't want to
blend class codes across the CRS transform.

In [ ]:
with rasterio.open(RASTER_TIF) as src:
    src_transform_decimated = src.transform * src.transform.scale(
        (src.width / out_w), (src.height / out_h)
    )
    dst_crs = "EPSG:4326"
    dst_transform, dst_w, dst_h = calculate_default_transform(
        src.crs, dst_crs, out_w, out_h,
        *rasterio.transform.array_bounds(out_h, out_w, src_transform_decimated)
    )

    classified_wgs84 = np.zeros((dst_h, dst_w), dtype=classified_src_crs.dtype)
    reproject(
        source=classified_src_crs,
        destination=classified_wgs84,
        src_transform=src_transform_decimated,
        src_crs=src.crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest
    )

    west, north = dst_transform * (0, 0)
    east, south = dst_transform * (dst_w, dst_h)
    bounds = [[west, north], [east, north], [east, south], [west, south]]

rgba = np.zeros((dst_h, dst_w, 4), dtype=np.uint8)
for cls, color in CLASS_COLORS.items():
    rgba[classified_wgs84 == cls] = color

png_path = f"{OUTPUT_DIR}/raster_overlay.png"
Image.fromarray(rgba, mode="RGBA").save(png_path)

bounds_path = f"{OUTPUT_DIR}/raster_bounds.json"
with open(bounds_path, "w") as f:
    json.dump(bounds, f)

print(f"Wrote {png_path} ({dst_w}x{dst_h}px)")
print(f"Wrote {bounds_path}")

## Step 6 — Preview the colorized overlay

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(rgba)
ax.set_title("Colorized classification overlay")
ax.set_axis_off()
handles = [plt.Rectangle((0,0),1,1, color=np.array(c)/255) for c in CLASS_COLORS.values()]
ax.legend(handles, [CLASS_LABELS[k] for k in CLASS_COLORS], loc="lower right", fontsize=9)
plt.show()

print("\nDone. Copy this notebook's OUTPUT_DIR contents into adu-viewer/data/ if not already there,")
print("then refresh index.html.")